In [40]:
from pyspark.sql.functions import broadcast, col, count, floor, round, when, countDistinct, regexp_replace, lower, trim, lpad 
from pyspark.sql.window import Window
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
)
import os
import sys

# Usar o Python do kernel tambem nos workers.
os.environ["PYSPARK_PYTHON"] = sys.executable

import pyspark
from pyspark.sql import SparkSession

print("Python:", sys.executable)
print("PySpark:", pyspark.__version__)

# Reinicie o kernel antes de criar a sessao com a nova memoria.
spark = (
    SparkSession.builder
    .master("local[1]")
    .appName("diagnostico")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

# print("Spark:", spark.version)
# spark.range(3).show()



Python: c:\Users\Andrey\AppData\Local\Programs\Python\Python313\python.exe
PySpark: 4.2.0


PEGANDO TODOS OS PARQUETS E CSV's


In [86]:
df_addresses = spark.read.parquet("C:\\Users\\Andrey\\OneDrive\\Documents\\GITHUB\\CantuStore-PROVA\\data\\raw\\tb_addresses.parquet")
df_cartentries = spark.read.parquet("C:\\Users\\Andrey\\OneDrive\\Documents\\GITHUB\\CantuStore-PROVA\\data\\raw\\tb_cartentries.parquet")
df_carts = spark.read.parquet("C:\\Users\\Andrey\\OneDrive\\Documents\\GITHUB\\CantuStore-PROVA\\data\\raw\\tb_carts.parquet")
df_payments = spark.read.parquet("C:\\Users\\Andrey\\OneDrive\\Documents\\GITHUB\\CantuStore-PROVA\\data\\raw\\tb_paymentinfos.parquet")

df_cmssitelp = (
    spark.read
    .option("header", True)
    .option("sep", "|")
    .csv(r"C:\Users\Andrey\OneDrive\Documents\GITHUB\CantuStore-PROVA\data\raw\tb_cmssitelp.csv")
)
df_regions = (
    spark.read
    .option("header", True)
    .option("sep", "|")
    .csv(r"C:\Users\Andrey\OneDrive\Documents\GITHUB\CantuStore-PROVA\data\raw\tb_regions.csv")
)
df_users = (
    spark.read
    .option("header", True)
    .option("sep", "|")
    .csv(r"C:\\Users\\Andrey\\OneDrive\\Documents\\GITHUB\\CantuStore-PROVA\\data\\raw\\tb_users.csv")
)

df_paymentmodes = (
    spark.read
    .option("header", True)
    .option("sep", "|")
    .csv(r"C:\\Users\\Andrey\\OneDrive\\Documents\\GITHUB\\CantuStore-PROVA\\data\\raw\\tb_paymentmodes.csv")
)

In [3]:
df_cartentries.printSchema()


root
 |-- hjmpTS: long (nullable = true)
 |-- createdTS: string (nullable = true)
 |-- modifiedTS: string (nullable = true)
 |-- TypePkString: long (nullable = true)
 |-- OwnerPkString: double (nullable = true)
 |-- PK: long (nullable = true)
 |-- sealed: double (nullable = true)
 |-- p_baseprice: double (nullable = true)
 |-- p_calculated: double (nullable = true)
 |-- p_entrynumber: double (nullable = true)
 |-- p_product: double (nullable = true)
 |-- p_quantity: double (nullable = true)
 |-- p_totalprice: double (nullable = true)
 |-- p_unit: double (nullable = true)
 |-- p_giveaway: double (nullable = true)
 |-- p_rejected: double (nullable = true)
 |-- p_entrygroupnumbers: double (nullable = true)
 |-- p_order: double (nullable = true)



In [4]:
df_carts.printSchema()

root
 |-- hjmpTS: double (nullable = true)
 |-- createdTS: string (nullable = true)
 |-- modifiedTS: string (nullable = true)
 |-- TypePkString: double (nullable = true)
 |-- OwnerPkString: double (nullable = true)
 |-- PK: long (nullable = true)
 |-- sealed: double (nullable = true)
 |-- p_calculated: double (nullable = true)
 |-- p_code: double (nullable = true)
 |-- p_currency: double (nullable = true)
 |-- p_deliveryaddress: double (nullable = true)
 |-- p_deliverycost: double (nullable = true)
 |-- p_deliverymode: double (nullable = true)
 |-- p_deliverystatus: double (nullable = true)
 |-- p_description: double (nullable = true)
 |-- p_expirationtime: double (nullable = true)
 |-- p_name: double (nullable = true)
 |-- p_net: double (nullable = true)
 |-- p_paymentaddress: double (nullable = true)
 |-- p_paymentcost: double (nullable = true)
 |-- p_paymentinfo: double (nullable = true)
 |-- p_paymentmode: double (nullable = true)
 |-- p_paymentstatus: double (nullable = true)
 |--

In [5]:
print("count of df_carts:", df_carts.count())
print("distintos:", df_cartentries.select("PK").distinct().count())
print("nulos:", df_cartentries.filter(col("PK").isNull()).count())


count of df_carts: 16047042
distintos: 2461265
nulos: 0


TRATAMENTO DF_CARTS

In [6]:
# Contar apenas PK evita ordenar todas as colunas na janela.
duplicatas_pk = (
    df_carts.groupBy("PK")
    .agg(count("PK").alias("count_PK"))
    .filter(col("count_PK") > 1)
)

# Recuperar as linhas completas de uma chave duplicada para a amostra.
(
    df_carts.filter(col('PK') == 9775464218667)
).show(truncate=False)


+------+-----------------------+-----------------------+----------------+-------------+-------------+------+------------+-----------+-----------------+-----------------+--------------+--------------+----------------+-------------+----------------+------+-----+----------------+-------------+-------------+-------------+---------------+--------+--------------+------------+------------+----------------+----------+-----------------+----------+------------------------------+-----------------------------+----------------------+----------------+-----------------+------------------------------------+--------------+-----------------+--------------------------+--------------------------+
|hjmpTS|createdTS              |modifiedTS             |TypePkString    |OwnerPkString|PK           |sealed|p_calculated|p_code     |p_currency       |p_deliveryaddress|p_deliverycost|p_deliverymode|p_deliverystatus|p_description|p_expirationtime|p_name|p_net|p_paymentaddress|p_paymentcost|p_paymentinfo|p_payment

In [7]:
# print("Tudo_distinct:", df_carts.distinct().count())
# print("apenas PK distinct:", df_carts.select("PK").distinct().count())

In [8]:
df_carts.groupBy("PK").agg(count("PK").alias("count_PK")).orderBy("count_PK", ascending=False).show(5, truncate=False)

+-------------+--------+
|PK           |count_PK|
+-------------+--------+
|9775464218667|2       |
|9777573658667|2       |
|9775504621611|2       |
|9776061349931|2       |
|9776527212587|2       |
+-------------+--------+
only showing top 5 rows


In [9]:
from pyspark.sql import Window
from pyspark.sql.functions import col, row_number, count

# Janela para numerar os registros de cada PK
window = Window.partitionBy("PK").orderBy(col("modifiedTS").desc())

# Adicionar row_number e quantidade de registros por PK
df_carts_rn = (
    df_carts
    .withColumn("rn", row_number().over(window))
    .withColumn("count_pk", count("*").over(Window.partitionBy("PK")))
)

# Manter os dois registros das PKs repetidas
df_carts_duplicadas = (
    df_carts_rn
    .filter(col("count_pk") > 1)
    .orderBy("PK", "rn")
)

# df_carts_duplicadas.show(20, truncate=False)

In [12]:
cep_tratado = (
    df_carts_rn
    .withColumn(
        "cep_tratado",
        when(
            col("p_zipcodecalculatedelivery").isNull() |
            (lower(trim(col("p_zipcodecalculatedelivery"))) == "nan") |
            (trim(col("p_zipcodecalculatedelivery")) == ""),
            None
        ).otherwise(
            col("p_zipcodecalculatedelivery")
        )
    )
    .withColumn(
        "cep_tratado",
        regexp_replace(col("cep_tratado"), r"\.0$", "")
    )
    .withColumn(
        "cep_tratado",
        when(
            col("cep_tratado").rlike(r"^[0-9]{1,8}$"),
            lpad(col("cep_tratado"), 8, "0")
        ).otherwise(col("cep_tratado"))
    )
)

In [13]:
cep_tratado.groupBy("PK").agg(
    countDistinct("cep_tratado").alias("count_cep")
).filter(
    col("count_cep") > 1
).orderBy(
    "count_cep", ascending=False
).show(20, truncate=False)

+---+---------+
|PK |count_cep|
+---+---------+
+---+---------+



In [14]:
df_validacao = cep_tratado.drop(
    "p_zipcodecalculatedelivery",
    "rn",
    "count_pk"
)

# print("Total:", df_validacao.count())

# print("Linhas distintas:", df_validacao.distinct().count())

# print("PKs distintas:", df_validacao.select("PK").distinct().count())

In [15]:
print("PKs com valor nulo:", cep_tratado.filter(col('PK').isNull()).count())
print("cep_tratado com valor nulo:", cep_tratado.filter(col('cep_tratado').isNull()).count())


PKs com valor nulo: 0
cep_tratado com valor nulo: 14202714


In [16]:
cep_dedup = df_validacao.dropDuplicates()

print("cep_invalido", cep_dedup.filter(~col("cep_tratado").rlike(r"^[0-9]{8}$")).count())

cep_invalido 9


In [17]:
cep_dedup.filter(~col("cep_tratado").rlike(r"^[0-9]{8}$")).show(20, truncate=False)

# 9 casos de cep invalidos que desconsiderarei

+------+-----------------------+-----------------------+----------------+-------------+--------------+------+------------+-----------+-----------------+-----------------+--------------+-----------------+----------------+-------------+----------------+------+-----+----------------+-------------+-----------------+-----------------+---------------+--------+--------------+------------+-----------------+----------------+----------+-----------------+-----------------+------------------------------+-----------------------------+----------------------+----------------+-----------------+------------------------------------+-----------------+-----------------+--------------------------+-----------+
|hjmpTS|createdTS              |modifiedTS             |TypePkString    |OwnerPkString|PK            |sealed|p_calculated|p_code     |p_currency       |p_deliveryaddress|p_deliverycost|p_deliverymode   |p_deliverystatus|p_description|p_expirationtime|p_name|p_net|p_paymentaddress|p_paymentcost|p_payme

In [42]:
from pyspark.sql import functions as F
from pyspark.sql import Window

df_percentual = (
    cep_dedup
    .groupBy("p_calculated")
    .agg(F.count("*").alias("quantidade"))
    .withColumn(
        "percentual",
        F.round(
            F.col("quantidade") * 100.0 /
            F.sum("quantidade").over(Window.partitionBy()),
            2
        )
    )
    .orderBy(F.col("quantidade").desc())
)

df_percentual.show(truncate=False)


#P_calculated pode indicar carrinho com algo dentro, mesmo que nao gere compra

+------------+----------+----------+
|p_calculated|quantidade|percentual|
+------------+----------+----------+
|0.0         |13470594  |84.0      |
|1.0         |2536556   |15.82     |
|NULL        |28758     |0.18      |
+------------+----------+----------+



In [18]:
cep_dedup.count() == cep_dedup.select("PK").distinct().count() #TERMINAMOS TODAS AS VALIDACOES, AGORA PODEMOS SEGUIR PARA O PROXIMO PASSO

True

TRATAMENTO CARTENTRIES

In [19]:
print(df_cartentries.count())
print(df_cartentries.select("PK").distinct().count())
print(df_cartentries.filter(col("PK").isNull()).count())



2461265
2461265
0


In [20]:
cart_cruza_cartentries = df_cartentries.alias("a").join(cep_dedup.alias('b'), on=[col("b.PK") == col("a.p_order")], how="inner")

cart_cruza_cartentries.count()

2461258

In [21]:
cep_dedup.groupBy("p_status").count().show(truncate=False)

+-----------------+--------+
|p_status         |count   |
+-----------------+--------+
|NULL             |16035902|
|8.796095578203E12|6       |
+-----------------+--------+



In [22]:
cep_dedup.groupBy("p_paymentstatus").count().show(truncate=False)

+---------------+--------+
|p_paymentstatus|count   |
+---------------+--------+
|NULL           |16035908|
+---------------+--------+



TRATAMENTO PAYMENT_INFOS


In [23]:
df_payments.printSchema()

root
 |-- createdTS: string (nullable = true)
 |-- modifiedTS: string (nullable = true)
 |-- PK: long (nullable = true)
 |-- p_installments: double (nullable = true)



In [24]:
df_payments.show(5, truncate=False)

+-----------------------+-----------------------+-------------+--------------+
|createdTS              |modifiedTS             |PK           |p_installments|
+-----------------------+-----------------------+-------------+--------------+
|2019-12-11 10:40:16.411|2019-12-11 12:05:08.081|8801041023018|NULL          |
|2019-12-11 10:40:16.587|2019-12-11 12:05:08.084|8801041055786|10.0          |
|2019-12-11 10:40:16.587|2019-12-11 12:05:08.087|8801041088554|10.0          |
|2019-12-11 10:40:18.097|2019-12-11 10:40:18.100|8801041121322|10.0          |
|2019-12-11 10:40:18.600|2019-12-11 10:40:18.610|8801041154090|1.0           |
+-----------------------+-----------------------+-------------+--------------+
only showing top 5 rows


In [27]:
print(cart_cruza_cartentries.filter(col("p_order").isNull()).count())
print(cep_dedup.filter(col("p_paymentinfo").isNull()).count())
print(cep_dedup.filter(~col("p_paymentinfo").isNull()).count())

0
14808798
1227110


In [30]:
df_payments.join(cep_dedup, on=df_payments.PK == cep_dedup.p_paymentinfo, how="leftanti").show(5, truncate=False)

+-----------------------+-----------------------+-------------+--------------+
|createdTS              |modifiedTS             |PK           |p_installments|
+-----------------------+-----------------------+-------------+--------------+
|2019-12-11 10:40:16.587|2019-12-11 12:05:08.084|8801041055786|10.0          |
|2019-12-11 10:40:19.498|2019-12-11 10:40:19.500|8801041219626|10.0          |
|2019-12-11 10:40:20.332|2019-12-11 10:40:20.334|8801041285162|1.0           |
|2019-12-11 10:40:20.763|2019-12-11 10:40:20.766|8801041317930|10.0          |
|2019-12-11 10:40:21.550|2019-12-11 10:40:21.552|8801041383466|10.0          |
+-----------------------+-----------------------+-------------+--------------+
only showing top 5 rows


In [32]:
from pyspark.sql.functions import col

# 1. Selecionar carrinhos que possuem referência de pagamento
df_carts_com_pagamento = cep_dedup.filter(
    col("p_paymentinfo").isNotNull()
)

# 2. Identificar carrinhos sem correspondência na tabela de pagamentos
df_pagamentos_nao_encontrados = (
    df_carts_com_pagamento.alias("a")
    .join(
        df_payments.alias("b"),
        on=col("a.p_paymentinfo") == col("b.PK"),
        how="left_anti"
    )
)

# 3. Verificar os resultados
print("Carrinhos com referência de pagamento:",
      df_carts_com_pagamento.count())

print("Carrinhos sem pagamento correspondente:",
      df_pagamentos_nao_encontrados.count())

# 4. Visualizar alguns registros sem correspondência
df_pagamentos_nao_encontrados.select(
    "PK",
    "p_paymentinfo",
    "createdTS",
    "modifiedTS"
).show(20, truncate=False)

Carrinhos com referência de pagamento: 1227110
Carrinhos sem pagamento correspondente: 650
+--------------+-----------------+-----------------------+-----------------------+
|PK            |p_paymentinfo    |createdTS              |modifiedTS             |
+--------------+-----------------+-----------------------+-----------------------+
|10279150190635|8.796256975621E12|2022-02-02 12:10:43.460|2022-02-10 10:09:27.849|
|11121161732139|8.798616304389E12|2022-06-09 13:54:32.331|2022-07-22 16:05:56.676|
|9819612053547 |8.799998130903E12|2021-06-02 16:51:10.675|2021-06-02 17:15:04.151|
|9819637645355 |8.800027523799E12|2021-06-02 16:06:47.344|2021-06-02 17:09:24.911|
|9820268757035 |8.800227539671E12|2021-06-05 07:07:24.582|2021-06-09 12:45:51.917|
|9822506254379 |8.800258013911E12|2021-06-11 11:43:26.745|2021-06-11 11:45:04.542|
|9843468337195 |8.801760983767E12|2021-08-18 10:03:30.641|2021-08-18 11:00:47.868|
|9864767766571 |8.804214520535E12|2021-10-23 17:03:57.523|2021-11-10 12:35:18.3

p_calculated = existencia de itens?


In [47]:
df_cartentries2 = df_cartentries.select("p_order").distinct()
df2 = cep_dedup.join(df_cartentries2, on=cep_dedup.PK == df_cartentries2.p_order, how="left")

df2.withColumn(
    'possui_produto', when(col("p_order").isNotNull(), 1).otherwise(0)
).groupBy("p_calculated", "possui_produto").agg(count("*").alias("quantidade")).show(truncate=False)


#+99% dos casos p_calculated = 0 não possuem produtos
#+80% dos casos p_calculated = 1 possuem produtos

+------------+--------------+----------+
|p_calculated|possui_produto|quantidade|
+------------+--------------+----------+
|1.0         |0             |492722    |
|NULL        |1             |28758     |
|0.0         |0             |13432895  |
|0.0         |1             |37699     |
|1.0         |1             |2043834   |
+------------+--------------+----------+



identificar produtos com mais abandonos

In [52]:


df_cmssitelp.printSchema()
df_cmssitelp.show(5, truncate=False)

root
 |-- ITEMPK: string (nullable = true)
 |-- p_name: string (nullable = true)

+-------------+------------------------+
|ITEMPK       |p_name                  |
+-------------+------------------------+
|8796093056040|PneuStore               |
|8796125824040|Pneu Store Mercado Livre|
|8796125856808|Pneu Store Cnova        |
|8796125889576|Pneu Store B2W          |
|8796125922344|Pneu Store Amazon       |
+-------------+------------------------+
only showing top 5 rows


verificar se p_ste do carts = itempk


In [55]:
from pyspark.sql import functions as F


df_sites = df_cmssitelp.select(
    F.col("ITEMPK").cast("long").alias("site_id"),
    F.col("p_name").alias("nome_site")
)

df_teste_site = (
    cep_dedup
    .withColumn("site_id", F.col("p_site").cast("long"))
    .join(df_sites, on="site_id", how="left")
)

df_teste_site.groupBy("nome_site").count().show(truncate=False)

+--------------------+--------+
|nome_site           |count   |
+--------------------+--------+
|NULL                |28771   |
|PneuStore           |13384901|
|Cantupneus Site     |96167   |
|Cantupneus Site NOVO|2526069 |
+--------------------+--------+



verificando addresses e regiões

In [56]:
df_addresses.printSchema()
df_regions.printSchema()

root
 |-- createdTS: string (nullable = true)
 |-- modifiedTS: string (nullable = true)
 |-- PK: long (nullable = true)
 |-- p_country: long (nullable = true)
 |-- p_district: string (nullable = true)
 |-- p_postalcode: string (nullable = true)
 |-- p_region: double (nullable = true)
 |-- p_town: string (nullable = true)
 |-- p_ibgecode: string (nullable = true)
 |-- p_reference: string (nullable = true)

root
 |-- hjmpTS: string (nullable = true)
 |-- createdTS: string (nullable = true)
 |-- modifiedTS: string (nullable = true)
 |-- TypePkString: string (nullable = true)
 |-- OwnerPkString: string (nullable = true)
 |-- PK: string (nullable = true)
 |-- sealed: string (nullable = true)
 |-- p_active: string (nullable = true)
 |-- p_isocode: string (nullable = true)
 |-- p_country: string (nullable = true)
 |-- p_isocodeshort: string (nullable = true)
 |-- aCLTS: string (nullable = true)
 |-- propTS: string (nullable = true)



cobertura de endereços


In [57]:
cep_dedup.agg(
    F.count("p_paymentaddress").alias("enderecos_pagamento"),
    F.count("p_deliveryaddress").alias("enderecos_entrega")
).show()

#poucos endereços de entrega, tratarei como possivel abandono de carrinho

+-------------------+-----------------+
|enderecos_pagamento|enderecos_entrega|
+-------------------+-----------------+
|                  6|           144732|
+-------------------+-----------------+



identificação de estados

In [58]:
df_regions.select(
    "PK",
    "p_isocode",
    "p_isocodeshort"
).show(10, truncate=False)

+-------------+---------+--------------+
|PK           |p_isocode|p_isocodeshort|
+-------------+---------+--------------+
|8796125823011|BR-SP    |SP            |
|8796158591011|BR-AP    |AP            |
|8796158623779|BR-AC    |AC            |
|8796158656547|BR-ES    |ES            |
|8796158689315|BR-DF    |DF            |
|8796158722083|BR-BA    |BA            |
|8796158754851|BR-AL    |AL            |
|8796158787619|BR-AM    |AM            |
|8796158820387|BR-CE    |CE            |
|8796158853155|BR-GO    |GO            |
+-------------+---------+--------------+
only showing top 10 rows


Definição de metricas de abandono de carrinho:

1- Obrigatoriamente precisa possuir item no carrinho
2- ainda analisando casos sem endereço e/ou sem pagamento, mas com itens no carrinho
3- se tivermos itens no carrinho + endereço + pagamento, considerarei compra efetuada

In [59]:

from pyspark.sql import functions as F
from pyspark.sql import Window

# 1. Criar indicadores de comportamento do carrinho
df_perfis = (
    df2
    .withColumn(
        "possui_produto",
        F.when(F.col("p_order").isNotNull(), 1).otherwise(0)
    )
    .withColumn(
        "possui_endereco",
        F.when(F.col("p_deliveryaddress").isNotNull(), 1).otherwise(0)
    )
    .withColumn(
        "possui_pagamento",
        F.when(F.col("p_paymentinfo").isNotNull(), 1).otherwise(0)
    )
)

# 2. Agrupar por perfil e calcular a participação percentual
df_resumo_perfis = (
    df_perfis
    .groupBy(
        "possui_produto",
        "possui_endereco",
        "possui_pagamento"
    )
    .agg(F.count("*").alias("quantidade"))
    .withColumn(
        "percentual",
        F.round(
            F.col("quantidade") * 100.0 /
            F.sum("quantidade").over(Window.partitionBy()),
            2
        )
    )
    .orderBy(F.col("quantidade").desc())
)

df_resumo_perfis.show(truncate=False)
#86% dos perfis não possuem produto no carrinho. O que faz sentido, pensando em um cenário real de um site


+--------------+---------------+----------------+----------+----------+
|possui_produto|possui_endereco|possui_pagamento|quantidade|percentual|
+--------------+---------------+----------------+----------+----------+
|0             |0              |0               |13856368  |86.41     |
|1             |0              |1               |1061925   |6.62      |
|1             |0              |0               |914063    |5.7       |
|1             |1              |1               |100392    |0.63      |
|0             |0              |1               |58820     |0.37      |
|1             |1              |0               |33911     |0.21      |
|0             |1              |1               |5973      |0.04      |
|0             |1              |0               |4456      |0.03      |
+--------------+---------------+----------------+----------+----------+



somente carrinhos com produto

In [60]:

df_perfis_com_produto = (
    df_perfis
    .filter(F.col("possui_produto") == 1)
    .groupBy("possui_endereco", "possui_pagamento")
    .agg(F.count("*").alias("quantidade"))
    .withColumn(
        "percentual",
        F.round(
            F.col("quantidade") * 100.0 /
            F.sum("quantidade").over(Window.partitionBy()),
            2
        )
    )
    .orderBy(F.col("quantidade").desc())
)

df_perfis_com_produto.show(truncate=False)


+---------------+----------------+----------+----------+
|possui_endereco|possui_pagamento|quantidade|percentual|
+---------------+----------------+----------+----------+
|0              |1               |1061925   |50.32     |
|0              |0               |914063    |43.31     |
|1              |1               |100392    |4.76      |
|1              |0               |33911     |1.61      |
+---------------+----------------+----------+----------+



UPLOAD DE DIMS E FATOS

In [63]:
from pyspark.sql import functions as F
from pyspark import StorageLevel
from datetime import datetime
from pathlib import Path
import json

# try_cast requer Spark 3.3 ou superior.
print("Versão do Spark:", spark.version)

def numero(coluna, tipo):
    return F.expr(f"try_cast(`{coluna}` as {tipo})")

def chave(coluna):
    # Evita truncar referências fracionárias.
    # As referências lidas como double precisam estar no intervalo exato.
    valor = numero(coluna, "decimal(38,6)")

    return F.when(
        (valor == F.floor(valor)) &
        (F.abs(valor) <= 9007199254740991),
        numero(coluna, "bigint")
    )

def validar_chave(df, coluna, nome):
    if df.filter(F.col(coluna).isNull()).limit(1).count():
        raise ValueError(f"{nome}: chave nula em {coluna}")

    if (
        df.groupBy(coluna)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    ):
        raise ValueError(f"{nome}: chave duplicada em {coluna}")

    print(f"{nome}: chave validada")

Versão do Spark: 4.2.0


In [64]:
itens_base = (
    df_cartentries
    .select(
        F.col("PK").alias("item_id"),
        chave("p_order").alias("carrinho_id"),
        chave("p_product").alias("produto_id_origem"),
        numero("p_quantity", "decimal(18,4)").alias("quantidade"),
        numero("p_baseprice", "decimal(20,4)").alias("preco_base"),
        numero("p_totalprice", "decimal(20,4)").alias("valor_item")
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

validar_chave(itens_base, "item_id", "itens_base")

carrinhos_base = (
    cep_dedup
    .select(
        F.col("PK").alias("carrinho_id"),
        F.to_date(
            F.expr("try_cast(createdTS as timestamp)")
        ).alias("data_criacao"),
        chave("p_site").alias("canal_id_origem"),
        chave("p_deliveryaddress").alias("endereco_id"),
        F.col("p_deliveryaddress").isNotNull()
            .cast("int").alias("possui_endereco"),
        F.col("p_paymentinfo").isNotNull()
            .cast("int").alias("possui_pagamento")
    )
)

validar_chave(carrinhos_base, "carrinho_id", "carrinhos_base")

# LEFT SEMI preserva somente carrinhos com registros de itens,
# sem multiplicar carrinhos pelo número de itens.
carrinhos_com_itens = (
    carrinhos_base
    .join(
        itens_base.select("carrinho_id").distinct(),
        on="carrinho_id",
        how="left_semi"
    )
)

itens_base: chave validada
carrinhos_base: chave validada


dimensão canais e regiao

In [65]:
dim_canal = (
    df_cmssitelp
    .select(
        chave("ITEMPK").alias("canal_id"),
        F.col("p_name").alias("nome_canal")
    )
)

validar_chave(dim_canal, "canal_id", "dim_canal")

dim_canal = dim_canal.unionByName(
    spark.createDataFrame(
        [(-1, "Canal não identificado")],
        "canal_id long, nome_canal string"
    )
)

dim_regiao = (
    df_regions
    .select(
        chave("PK").alias("regiao_id"),
        F.col("p_isocode").alias("codigo_regiao"),
        F.upper(F.trim(F.col("p_isocodeshort"))).alias("uf")
    )
)

validar_chave(dim_regiao, "regiao_id", "dim_regiao")

dim_regiao = dim_regiao.unionByName(
    spark.createDataFrame(
        [(-1, "Não identificada", "NI")],
        "regiao_id long, codigo_regiao string, uf string"
    )
)

enderecos_base = (
    df_addresses
    .select(
        F.col("PK").alias("endereco_id"),
        chave("p_region").alias("regiao_id_origem")
    )
)

validar_chave(enderecos_base, "endereco_id", "enderecos_base")

dim_canal: chave validada
dim_regiao: chave validada
enderecos_base: chave validada


Dimensão carrinho e perfil de checkout

In [66]:
dim_carrinho = (
    carrinhos_com_itens

    .join(
        enderecos_base,
        on="endereco_id",
        how="left"
    )

    .join(
        dim_canal.select(
            F.col("canal_id").alias("canal_id_origem"),
            "canal_id"
        ),
        on="canal_id_origem",
        how="left"
    )

    .join(
        dim_regiao.select(
            F.col("regiao_id").alias("regiao_id_origem"),
            "regiao_id"
        ),
        on="regiao_id_origem",
        how="left"
    )

    .withColumn(
        "canal_id",
        F.coalesce(F.col("canal_id"), F.lit(-1)).cast("long")
    )

    .withColumn(
        "regiao_id",
        F.coalesce(F.col("regiao_id"), F.lit(-1)).cast("long")
    )

    .withColumn(
        "data_id",
        F.coalesce(
            F.date_format("data_criacao", "yyyyMMdd").cast("int"),
            F.lit(-1)
        )
    )

    .withColumn(
        "perfil_checkout",
        F.when(
            (F.col("possui_endereco") == 1) &
            (F.col("possui_pagamento") == 1),
            "Com endereço e pagamento"
        )
        .when(
            F.col("possui_endereco") == 1,
            "Com endereço, sem pagamento"
        )
        .when(
            F.col("possui_pagamento") == 1,
            "Com pagamento, sem endereço"
        )
        .otherwise("Sem endereço e pagamento")
    )

    .select(
        "carrinho_id",
        "data_id",
        "data_criacao",
        "canal_id",
        "regiao_id",
        "possui_endereco",
        "possui_pagamento",
        "perfil_checkout"
    )

    .persist(StorageLevel.MEMORY_AND_DISK)
)

validar_chave(dim_carrinho, "carrinho_id", "dim_carrinho")

dim_carrinho.groupBy("perfil_checkout").count().show(truncate=False)

dim_carrinho: chave validada
+---------------------------+-------+
|perfil_checkout            |count  |
+---------------------------+-------+
|Com endereço, sem pagamento|33911  |
|Com endereço e pagamento   |100392 |
|Sem endereço e pagamento   |914063 |
|Com pagamento, sem endereço|1061925|
+---------------------------+-------+



Fato de itens

In [67]:
fato_itens_carrinho = (
    itens_base

    .join(
        dim_carrinho.select("carrinho_id"),
        on="carrinho_id",
        how="left_semi"
    )

    .withColumn(
        "produto_id",
        F.coalesce(F.col("produto_id_origem"), F.lit(-1)).cast("long")
    )

    .withColumn(
        "produto_identificado",
        F.col("produto_id_origem").isNotNull().cast("int")
    )

    .withColumn(
        "quantidade_invalida",
        (
            F.col("quantidade").isNull() |
            (F.col("quantidade") <= 0)
        ).cast("int")
    )

    .withColumn(
        "valor_invalido",
        (
            F.col("valor_item").isNull() |
            (F.col("valor_item") < 0)
        ).cast("int")
    )

    .select(
        "item_id",
        "carrinho_id",
        "produto_id",
        "produto_identificado",
        "quantidade",
        "preco_base",
        "valor_item",
        "quantidade_invalida",
        "valor_invalido"
    )

    .persist(StorageLevel.MEMORY_AND_DISK)
)

fato_itens_carrinho.show(5, truncate=False)

+-------------+-------------+-------------+--------------------+----------+----------+----------+-------------------+--------------+
|item_id      |carrinho_id  |produto_id   |produto_identificado|quantidade|preco_base|valor_item|quantidade_invalida|valor_invalido|
+-------------+-------------+-------------+--------------------+----------+----------+----------+-------------------+--------------+
|8887430774828|9335274766379|8800488456193|1                   |1.0000    |1278.4950 |1278.4950 |0                  |0             |
|8901070159916|9335274766379|8797318053889|1                   |1.0000    |1627.2000 |1627.2000 |0                  |0             |
|8827044003884|9383933149227|8798714658817|1                   |1.0000    |2519.5500 |2519.5500 |0                  |0             |
|8833735917612|9466025115691|8797307666433|1                   |5.0000    |1818.6750 |9093.3750 |0                  |0             |
|8931496755244|9470237409323|8797291839489|1                   |2.000

Dimensão de produto e data

In [68]:
dim_produto = (
    fato_itens_carrinho
    .join(
        dim_carrinho.select("carrinho_id", "data_criacao"),
        on="carrinho_id",
        how="inner"
    )
    .groupBy("produto_id")
    .agg(
        F.min("data_criacao").alias("primeira_aparicao_observada")
    )
    .withColumn(
        "rotulo_produto",
        F.when(
            F.col("produto_id") == -1,
            "Produto não identificado"
        ).otherwise(
            F.concat(
                F.lit("Produto "),
                F.col("produto_id").cast("string")
            )
        )
    )
)

limites_datas = (
    dim_carrinho
    .agg(
        F.min("data_criacao").alias("data_inicial"),
        F.max("data_criacao").alias("data_final")
    )
    .first()
)

if limites_datas["data_inicial"] is None:
    raise ValueError("Nenhuma data válida encontrada nos carrinhos com itens.")

dim_data = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(limites_datas["data_inicial"]),
                F.lit(limites_datas["data_final"]),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("data")
    )
    .withColumn(
        "data_id",
        F.date_format("data", "yyyyMMdd").cast("int")
    )
    .withColumn("ano", F.year("data"))
    .withColumn("mes", F.month("data"))
    .withColumn("dia", F.dayofmonth("data"))
    .withColumn("ano_mes", F.date_format("data", "yyyy-MM"))
    .withColumn(
        "ano_mes_ordem",
        F.col("ano") * 100 + F.col("mes")
    )
    .select(
        "data_id", "data", "ano", "mes",
        "dia", "ano_mes", "ano_mes_ordem"
    )
)

# Registro para carrinhos cuja data não está disponível.
dim_data = dim_data.unionByName(
    spark.createDataFrame(
        [(-1, None, None, None, None, "Sem data", None)],
        """
        data_id int, data date, ano int, mes int,
        dia int, ano_mes string, ano_mes_ordem int
        """
    )
)

dim_produto.show(5, truncate=False)
dim_data.orderBy("data_id").show(5, truncate=False)

+-------------+---------------------------+---------------------+
|produto_id   |primeira_aparicao_observada|rotulo_produto       |
+-------------+---------------------------+---------------------+
|8798714658817|2020-05-14                 |Produto 8798714658817|
|8797185703937|2020-11-15                 |Produto 8797185703937|
|8797306781697|2020-09-07                 |Produto 8797306781697|
|8801565376513|2020-07-20                 |Produto 8801565376513|
|8801370996737|2020-10-28                 |Produto 8801370996737|
+-------------+---------------------------+---------------------+
only showing top 5 rows
+--------+----------+----+----+----+--------+-------------+
|data_id |data      |ano |mes |dia |ano_mes |ano_mes_ordem|
+--------+----------+----+----+----+--------+-------------+
|-1      |NULL      |NULL|NULL|NULL|Sem data|NULL         |
|20191216|2019-12-16|2019|12  |16  |2019-12 |201912       |
|20191217|2019-12-17|2019|12  |17  |2019-12 |201912       |
|20191218|2019-12-18|2

fato dupla de produtos

In [69]:
produtos_por_carrinho = (
    fato_itens_carrinho
    .filter(F.col("produto_id") != -1)
    .select("carrinho_id", "produto_id")
    .distinct()
)

fato_duplas_carrinho = (
    produtos_por_carrinho.alias("a")
    .join(
        produtos_por_carrinho.alias("b"),
        (
            F.col("a.carrinho_id") == F.col("b.carrinho_id")
        ) & (
            F.col("a.produto_id") < F.col("b.produto_id")
        ),
        how="inner"
    )
    .select(
        F.col("a.carrinho_id").alias("carrinho_id"),
        F.col("a.produto_id").alias("produto_a_id"),
        F.col("b.produto_id").alias("produto_b_id")
    )
    .withColumn(
        "dupla",
        F.concat_ws(
            " + ",
            F.col("produto_a_id").cast("string"),
            F.col("produto_b_id").cast("string")
        )
    )
)

fato_duplas_carrinho.show(5, truncate=False)

+-------------+-------------+-------------+-----------------------------+
|carrinho_id  |produto_a_id |produto_b_id |dupla                        |
+-------------+-------------+-------------+-----------------------------+
|9335274766379|8797318053889|8800488456193|8797318053889 + 8800488456193|
|9595688845355|8797234790401|8800681689089|8797234790401 + 8800681689089|
|9765148524587|8797147463681|8798191550465|8797147463681 + 8798191550465|
|9765148524587|8797147463681|8798714560513|8797147463681 + 8798714560513|
|9765148524587|8798191550465|8798714560513|8798191550465 + 8798714560513|
+-------------+-------------+-------------+-----------------------------+
only showing top 5 rows


conciliação

In [70]:
tabelas = {
    "dim_canal": dim_canal,
    "dim_regiao": dim_regiao,
    "dim_carrinho": dim_carrinho,
    "fato_itens_carrinho": fato_itens_carrinho,
    "dim_produto": dim_produto,
    "dim_data": dim_data,
    "fato_duplas_carrinho": fato_duplas_carrinho
}

for nome, coluna in [
    ("dim_canal", "canal_id"),
    ("dim_regiao", "regiao_id"),
    ("dim_produto", "produto_id"),
    ("dim_data", "data_id")
]:
    validar_chave(tabelas[nome], coluna, nome)

# Verificar integridade referencial.
relacoes = [
    (fato_itens_carrinho, dim_carrinho, "carrinho_id"),
    (fato_itens_carrinho, dim_produto, "produto_id"),
    (dim_carrinho, dim_data, "data_id"),
    (dim_carrinho, dim_canal, "canal_id"),
    (dim_carrinho, dim_regiao, "regiao_id")
]

for origem, dimensao, coluna in relacoes:
    sem_correspondencia = (
        origem.select(coluna).distinct()
        .join(
            dimensao.select(coluna),
            on=coluna,
            how="left_anti"
        )
    )

    if sem_correspondencia.limit(1).count():
        raise ValueError(f"Referência sem dimensão: {coluna}")

total_itens_origem = itens_base.count()
total_itens_fato = fato_itens_carrinho.count()

total_itens_esperado = (
    itens_base
    .join(
        dim_carrinho.select("carrinho_id"),
        on="carrinho_id",
        how="left_semi"
    )
    .count()
)

assert total_itens_fato == total_itens_esperado, "Divergência na quantidade de itens"

total_carrinhos = dim_carrinho.count()

assert (
    fato_itens_carrinho.select("carrinho_id").distinct().count()
    == total_carrinhos
), "Divergência na quantidade de carrinhos"

qualidade_itens = (
    fato_itens_carrinho
    .agg(
        F.sum(
            F.when(F.col("produto_id") == -1, 1).otherwise(0)
        ).alias("itens_sem_produto"),
        F.sum("quantidade_invalida").alias("itens_quantidade_invalida"),
        F.sum("valor_invalido").alias("itens_valor_invalido")
    )
    .first()
    .asDict()
)

qualidade_carrinhos = (
    dim_carrinho
    .agg(
        F.sum(
            F.when(F.col("data_id") == -1, 1).otherwise(0)
        ).alias("carrinhos_sem_data"),
        F.sum(
            F.when(F.col("regiao_id") == -1, 1).otherwise(0)
        ).alias("carrinhos_sem_regiao"),
        F.sum(
            F.when(F.col("canal_id") == -1, 1).otherwise(0)
        ).alias("carrinhos_sem_canal")
    )
    .first()
    .asDict()
)

auditoria = {
    "itens_origem": total_itens_origem,
    "itens_fato": total_itens_fato,
    "itens_sem_carrinho_correspondente":
        total_itens_origem - total_itens_fato,
    "carrinhos_com_itens": total_carrinhos,
    **qualidade_itens,
    **qualidade_carrinhos
}

print(json.dumps(auditoria, indent=2, ensure_ascii=False))

print(
    "\nTotal de carrinhos observado anteriormente: 2.110.291"
    "\nCompare com o resultado atual; diferenças devem ser investigadas."
)

dim_canal: chave validada
dim_regiao: chave validada
dim_produto: chave validada
dim_data: chave validada
{
  "itens_origem": 2461265,
  "itens_fato": 2461258,
  "itens_sem_carrinho_correspondente": 7,
  "carrinhos_com_itens": 2110291,
  "itens_sem_produto": 1,
  "itens_quantidade_invalida": 0,
  "itens_valor_invalido": 0,
  "carrinhos_sem_data": 28758,
  "carrinhos_sem_regiao": 1976326,
  "carrinhos_sem_canal": 28758
}

Total de carrinhos observado anteriormente: 2.110.291
Compare com o resultado atual; diferenças devem ser investigadas.


exportação BI

In [73]:
import csv
import json
from pathlib import Path
from datetime import datetime

PASTA_BASE = Path(
    r"C:\Users\Andrey\OneDrive\Documents\GITHUB"
    r"\CantuStore-PROVA\data\processed"
)

PASTA_EXPORTACAO = (
    PASTA_BASE /
    datetime.now().strftime("modelo_python_%Y%m%d_%H%M%S_%f")
)

PASTA_EXPORTACAO.mkdir(parents=True, exist_ok=False)

resumo_exportacao = {}

for nome, df in tabelas.items():

    tabela_grande = (
        nome.startswith("fato_") or nome == "dim_carrinho"
    )

    df_exportacao = df.repartition(64) if tabela_grande else df

    arquivo_final = PASTA_EXPORTACAO / f"{nome}.csv"
    arquivo_temporario = PASTA_EXPORTACAO / f"{nome}.csv.partial"

    print(f"Iniciando: {nome}", flush=True)

    quantidade = 0

    with arquivo_temporario.open(
        mode="w",
        encoding="utf-8-sig",
        newline=""
    ) as arquivo:

        writer = csv.writer(arquivo, delimiter=";")
        writer.writerow(df.columns)

        for linha in df_exportacao.toLocalIterator(
            prefetchPartitions=False
        ):
            writer.writerow([
                "" if valor is None else valor
                for valor in linha
            ])

            quantidade += 1

            if quantidade % 250_000 == 0:
                print(
                    f"  {quantidade:,} registros gravados",
                    flush=True
                )

    # Só recebe extensão final após concluir a tabela.
    arquivo_temporario.rename(arquivo_final)

    resumo_exportacao[nome] = quantidade

    print(
        f"Concluída: {nome} — {quantidade:,} registros",
        flush=True
    )

(PASTA_EXPORTACAO / "auditoria.json").write_text(
    json.dumps(
        {
            "auditoria_modelo": auditoria,
            "linhas_exportadas": resumo_exportacao
        },
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

itens_base.unpersist()
dim_carrinho.unpersist()
fato_itens_carrinho.unpersist()

print(f"\nExportação concluída:\n{PASTA_EXPORTACAO}")

Iniciando: dim_canal
Concluída: dim_canal — 27 registros
Iniciando: dim_regiao
Concluída: dim_regiao — 29 registros
Iniciando: dim_carrinho
  250,000 registros gravados
  500,000 registros gravados
  750,000 registros gravados
  1,000,000 registros gravados
  1,250,000 registros gravados
  1,500,000 registros gravados
  1,750,000 registros gravados
  2,000,000 registros gravados
Concluída: dim_carrinho — 2,110,291 registros
Iniciando: fato_itens_carrinho
  250,000 registros gravados
  500,000 registros gravados
  750,000 registros gravados
  1,000,000 registros gravados
  1,250,000 registros gravados
  1,500,000 registros gravados
  1,750,000 registros gravados
  2,000,000 registros gravados
  2,250,000 registros gravados
Concluída: fato_itens_carrinho — 2,461,258 registros
Iniciando: dim_produto
Concluída: dim_produto — 5,505 registros
Iniciando: dim_data
Concluída: dim_data — 955 registros
Iniciando: fato_duplas_carrinho
  250,000 registros gravados
Concluída: fato_duplas_carrinho — 

In [77]:
import runpy

config = runpy.run_path(r"C:\Users\Andrey\OneDrive\Documents\GITHUB\CantuStore-PROVA\CantuStore_PowerBI\configurar.py")
config["configurar"]()


config["configurar"](
    pasta_dados=r"C:\Users\Andrey\OneDrive\Documents\GITHUB\CantuStore-PROVA\data\processed\modelo_python_20261008_050336_103641"
)

Fontes configuradas: C:\Users\Andrey\OneDrive\Documents\GITHUB\CantuStore-PROVA\data\processed\modelo_python_20261008_050336_103641
Abra: C:\Users\Andrey\OneDrive\Documents\GITHUB\CantuStore-PROVA\CantuStore_PowerBI\CantuStore.pbip
Depois clique em Atualizar no Power BI Desktop.
Fontes configuradas: C:\Users\Andrey\OneDrive\Documents\GITHUB\CantuStore-PROVA\data\processed\modelo_python_20261008_050336_103641
Abra: C:\Users\Andrey\OneDrive\Documents\GITHUB\CantuStore-PROVA\CantuStore_PowerBI\CantuStore.pbip
Depois clique em Atualizar no Power BI Desktop.


'C:\\Users\\Andrey\\OneDrive\\Documents\\GITHUB\\CantuStore-PROVA\\data\\processed\\modelo_python_20261008_050336_103641'

In [87]:
bases = [
	("df_addresses", df_addresses),
	("df_cartentries", df_cartentries),
	("df_carts", df_carts),
	("df_payments", df_payments),
	("df_cmssitelp", df_cmssitelp),
	("df_paymentmodes", df_paymentmodes),
	("df_regions", df_regions),
	("df_users", df_users),
]

for nome, base in bases:
	print(f"\nSchema de {nome}:")
	base.show(20, truncate=False)


Schema de df_addresses:
+-----------------------+-----------------------+-------------+-------------+----------+------------+-----------------+-------+----------+-----------+
|createdTS              |modifiedTS             |PK           |p_country    |p_district|p_postalcode|p_region         |p_town |p_ibgecode|p_reference|
+-----------------------+-----------------------+-------------+-------------+----------+------------+-----------------+-------+----------+-----------+
|2019-06-25 14:23:12.630|2021-09-14 12:58:51.053|8796093186071|8796094005282|NULL      |nan         |NULL             |NULL   |nan       |NULL       |
|2019-07-18 10:16:55.213|2019-07-18 10:16:55.213|8796125888535|8796094005282|NULL      |09910580    |NULL             |Diadema|nan       |NULL       |
|2019-09-18 14:00:46.753|2019-09-18 14:00:46.753|8797928226839|8796094005282|Centro    |10120-080   |8.796159115299E12|Maringá|nan       |NULL       |
|2019-09-18 14:11:09.715|2019-09-18 14:11:09.715|8797928423447|879609